로컬 Neo4j에 테스트 데이터를 올려서 만드는 임시 graphDB(실제 데이터 아님)

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from IPython.display import display


# ============================================================
# 1. .env 환경변수 불러오기
# ============================================================

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")

NEO4J_USERNAME = (
    os.getenv("NEO4J_USERNAME")
    or os.getenv("NEO4J_USER")
)

NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

NEO4J_DATABASE = os.getenv(
    "NEO4J_DATABASE",
    "neo4j",
)


required_env = {
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USERNAME": NEO4J_USERNAME,
    "NEO4J_PASSWORD": NEO4J_PASSWORD,
}

missing = [
    key
    for key, value in required_env.items()
    if not value
]

if missing:
    raise ValueError(
        f".env에 다음 환경변수가 없습니다: "
        f"{', '.join(missing)}"
    )


# ============================================================
# 2. Neo4j 연결
# ============================================================

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(
        NEO4J_USERNAME,
        NEO4J_PASSWORD,
    ),
)

driver.verify_connectivity()

print("Neo4j 연결 성공")
print(f"URI      : {NEO4J_URI}")
print(f"Database : {NEO4J_DATABASE}")


# ============================================================
# 3. 적재할 샘플 데이터
# ============================================================
#
# 실제 데이터 적재 형태를 흉내 낸 가상 기업 데이터
#
# 온톨로지에서 정의한 속성만 사용:
#
# ParentCompany
#   -> crno, name, address
#
# SubsidiaryCompany
#   -> name, address
#
# Region
#   -> name
#
# Industry
#   -> name
#
# ============================================================


parent_companies = [
    {
        "crno": "1101110001001",
        "name": "가온홀딩스",
        "address": "서울특별시 강남구 테헤란로 120",
        "region": "서울특별시",
    },
    {
        "crno": "1101110001002",
        "name": "가온전자",
        "address": "경기도 성남시 분당구 판교로 210",
        "region": "경기도",
    },
    {
        "crno": "1601110001003",
        "name": "새빛테크",
        "address": "대전광역시 유성구 테크노로 80",
        "region": "대전광역시",
    },
]


subsidiaries = [
    {
        "name": "가온디지털",
        "address": "서울특별시 영등포구 국제금융로 30",
        "parent": "가온홀딩스",
        "region": "서울특별시",
        "industry": "IT서비스",
    },
    {
        "name": "가온모빌리티",
        "address": "경기도 화성시 산업로 150",
        "parent": "가온전자",
        "region": "경기도",
        "industry": "자동차부품",
    },
    {
        "name": "새빛데이터",
        "address": "대전광역시 유성구 대학로 90",
        "parent": "새빛테크",
        "region": "대전광역시",
        "industry": "데이터서비스",
    },
    {
        "name": "새빛에너지",
        "address": "충청남도 천안시 서북구 산업로 75",
        "parent": "새빛테크",
        "region": "충청남도",
        "industry": "에너지",
    },
]


# 모기업 ↔ 모기업 계열 관계
affiliations = [
    {
        "source": "가온홀딩스",
        "target": "가온전자",
    },
]


# ============================================================
# 4. 기존 Subsidiary 라벨 변환
# ============================================================
#
# 이전 코드에서 Subsidiary라는 라벨로 적재된 노드가 있다면
# 온톨로지의 SubsidiaryCompany로 변경합니다.
#
# 기존 노드의 속성과 관계는 그대로 유지됩니다.
#
# ============================================================


def migrate_old_subsidiary_label(driver):

    with driver.session(
        database=NEO4J_DATABASE
    ) as session:

        # 현재 DB의 라벨 확인
        result = session.run(
            """
            CALL db.labels()
            YIELD label
            RETURN label
            """
        )

        existing_labels = {
            record["label"]
            for record in result
        }

        # Subsidiary가 없으면 변환하지 않음
        if "Subsidiary" not in existing_labels:
            return 0

        # 기존 라벨 변경
        result = session.run(
            """
            MATCH (s:Subsidiary)

            SET s:SubsidiaryCompany
            REMOVE s:Subsidiary

            RETURN count(s) AS migrated_count
            """
        )

        record = result.single()

        return record["migrated_count"]


migrated_count = migrate_old_subsidiary_label(
    driver
)

if migrated_count > 0:
    print(
        f"\n기존 Subsidiary 노드 "
        f"{migrated_count}개를 "
        f"SubsidiaryCompany로 변경했습니다."
    )


# ============================================================
# 5. Neo4j 적재 함수
# ============================================================


def load_graph_data(driver):

    with driver.session(
        database=NEO4J_DATABASE
    ) as session:

        # ----------------------------------------------------
        # ParentCompany 생성
        #
        # ParentCompany -> Region
        # ----------------------------------------------------

        session.run(
            """
            UNWIND $rows AS row

            MERGE (p:ParentCompany {
                crno: row.crno
            })

            SET
                p.name = row.name,
                p.address = row.address

            MERGE (r:Region {
                name: row.region
            })

            MERGE (p)-[:LOCATED_IN]->(r)
            """,
            rows=parent_companies,
        )


        # ----------------------------------------------------
        # SubsidiaryCompany 생성
        #
        # ParentCompany -> SubsidiaryCompany
        # SubsidiaryCompany -> Region
        # SubsidiaryCompany -> Industry
        # ----------------------------------------------------

        session.run(
            """
            UNWIND $rows AS row

            MERGE (s:SubsidiaryCompany {
                name: row.name
            })

            SET
                s.address = row.address

            MERGE (r:Region {
                name: row.region
            })

            MERGE (i:Industry {
                name: row.industry
            })

            WITH
                row,
                s,
                r,
                i

            MATCH (p:ParentCompany {
                name: row.parent
            })

            MERGE
                (p)-[:HAS_SUBSIDIARY]->(s)

            MERGE
                (s)-[:LOCATED_IN]->(r)

            MERGE
                (s)-[:RELATED_TO_INDUSTRY]->(i)
            """,
            rows=subsidiaries,
        )


        # ----------------------------------------------------
        # ParentCompany -> ParentCompany
        #
        # 계열 관계
        # ----------------------------------------------------

        session.run(
            """
            UNWIND $rows AS row

            MATCH (source:ParentCompany {
                name: row.source
            })

            MATCH (target:ParentCompany {
                name: row.target
            })

            MERGE
                (source)-[:AFFILIATED_WITH]->(target)
            """,
            rows=affiliations,
        )


# ============================================================
# 6. 데이터 적재
# ============================================================

load_graph_data(driver)

print("\n데이터 적재 완료")


# ============================================================
# 7. 노드 확인
# ============================================================

node_query = """
MATCH (n)

WHERE
    n.name IN [
        "가온홀딩스",
        "가온전자",
        "새빛테크",
        "가온디지털",
        "가온모빌리티",
        "새빛데이터",
        "새빛에너지",
        "서울특별시",
        "경기도",
        "대전광역시",
        "충청남도",
        "IT서비스",
        "자동차부품",
        "데이터서비스",
        "에너지"
    ]

RETURN
    labels(n)[0] AS label,
    n.name AS name,
    n.crno AS crno,
    n.address AS address

ORDER BY
    label,
    name
"""


with driver.session(
    database=NEO4J_DATABASE
) as session:

    result = session.run(node_query)

    node_rows = [
        dict(record)
        for record in result
    ]


node_df = pd.DataFrame(
    node_rows
)


print("\n====================================")
print("[생성된 노드]")
print("====================================")

display(node_df)


# ============================================================
# 8. 레이블별 노드 확인
# ============================================================

label_query = """
MATCH (n)

WHERE
    n.name IN [
        "가온홀딩스",
        "가온전자",
        "새빛테크",
        "가온디지털",
        "가온모빌리티",
        "새빛데이터",
        "새빛에너지",
        "서울특별시",
        "경기도",
        "대전광역시",
        "충청남도",
        "IT서비스",
        "자동차부품",
        "데이터서비스",
        "에너지"
    ]

UNWIND labels(n) AS label

RETURN
    label,
    count(*) AS count

ORDER BY
    label
"""


with driver.session(
    database=NEO4J_DATABASE
) as session:

    result = session.run(label_query)

    label_rows = [
        dict(record)
        for record in result
    ]


label_df = pd.DataFrame(
    label_rows
)


print("\n====================================")
print("[레이블별 노드 수]")
print("====================================")

display(label_df)


# ============================================================
# 9. 관계 전체 확인
# ============================================================

relation_query = """
MATCH (a)-[r]->(b)

WHERE
    a.name IN [
        "가온홀딩스",
        "가온전자",
        "새빛테크",
        "가온디지털",
        "가온모빌리티",
        "새빛데이터",
        "새빛에너지"
    ]

RETURN
    labels(a)[0] AS source_label,
    a.name AS source,

    type(r) AS relation,

    labels(b)[0] AS target_label,
    b.name AS target

ORDER BY
    source,
    relation,
    target
"""


with driver.session(
    database=NEO4J_DATABASE
) as session:

    result = session.run(
        relation_query
    )

    relation_rows = [
        dict(record)
        for record in result
    ]


relation_df = pd.DataFrame(
    relation_rows
)


print("\n====================================")
print("[생성된 관계]")
print("====================================")

display(relation_df)


# ============================================================
# 10. 관계 타입별 개수
# ============================================================

relation_count_query = """
MATCH (a)-[r]->(b)

WHERE
    a.name IN [
        "가온홀딩스",
        "가온전자",
        "새빛테크",
        "가온디지털",
        "가온모빌리티",
        "새빛데이터",
        "새빛에너지"
    ]

RETURN
    type(r) AS relation,
    count(*) AS count

ORDER BY
    relation
"""


with driver.session(
    database=NEO4J_DATABASE
) as session:

    result = session.run(
        relation_count_query
    )

    relation_count_rows = [
        dict(record)
        for record in result
    ]


relation_count_df = pd.DataFrame(
    relation_count_rows
)


print("\n====================================")
print("[관계 타입별 개수]")
print("====================================")

display(relation_count_df)


# ============================================================
# 11. 온톨로지 규칙 검사
# ============================================================
#
# 허용된 관계:
#
# ParentCompany
#   -> AFFILIATED_WITH
#   -> ParentCompany
#
# ParentCompany
#   -> HAS_SUBSIDIARY
#   -> SubsidiaryCompany
#
# ParentCompany
#   -> LOCATED_IN
#   -> Region
#
# SubsidiaryCompany
#   -> LOCATED_IN
#   -> Region
#
# SubsidiaryCompany
#   -> RELATED_TO_INDUSTRY
#   -> Industry
#
# ============================================================


validation_query = """
MATCH (a)-[r]->(b)

WHERE
    a.name IN [
        "가온홀딩스",
        "가온전자",
        "새빛테크",
        "가온디지털",
        "가온모빌리티",
        "새빛데이터",
        "새빛에너지"
    ]

WITH
    labels(a)[0] AS source_label,
    type(r) AS relation,
    labels(b)[0] AS target_label,
    a,
    b

WITH
    source_label,
    relation,
    target_label,
    a,
    b,

    CASE

        WHEN
            source_label = "ParentCompany"
            AND relation = "AFFILIATED_WITH"
            AND target_label = "ParentCompany"
        THEN true

        WHEN
            source_label = "ParentCompany"
            AND relation = "HAS_SUBSIDIARY"
            AND target_label = "SubsidiaryCompany"
        THEN true

        WHEN
            source_label = "ParentCompany"
            AND relation = "LOCATED_IN"
            AND target_label = "Region"
        THEN true

        WHEN
            source_label = "SubsidiaryCompany"
            AND relation = "LOCATED_IN"
            AND target_label = "Region"
        THEN true

        WHEN
            source_label = "SubsidiaryCompany"
            AND relation = "RELATED_TO_INDUSTRY"
            AND target_label = "Industry"
        THEN true

        ELSE false

    END AS valid

RETURN
    a.name AS source,
    source_label,
    relation,
    b.name AS target,
    target_label,
    valid

ORDER BY
    valid,
    source,
    relation
"""


with driver.session(
    database=NEO4J_DATABASE
) as session:

    result = session.run(
        validation_query
    )

    validation_rows = [
        dict(record)
        for record in result
    ]


validation_df = pd.DataFrame(
    validation_rows
)


print("\n====================================")
print("[온톨로지 관계 검증]")
print("====================================")

display(validation_df)


# ============================================================
# 12. 최종 결과 요약
# ============================================================

invalid_count = 0

if not validation_df.empty:
    invalid_count = (
        validation_df["valid"] == False
    ).sum()


print("\n====================================")
print("적재 결과")
print("====================================")

print(
    f"확인된 노드 수 : "
    f"{len(node_df)}"
)

print(
    f"확인된 관계 수 : "
    f"{len(relation_df)}"
)

print()


if invalid_count == 0:
    print(
        "모든 관계가 현재 온톨로지 규칙을 "
        "만족합니다."
    )

else:
    print(
        f"온톨로지 규칙 위반 관계: "
        f"{invalid_count}개"
    )


print("\n[사용된 노드 레이블]")

for row in label_rows:

    print(
        f"- {row['label']}: "
        f"{row['count']}개"
    )


print("\n[사용된 관계]")

for row in relation_count_rows:

    print(
        f"- {row['relation']}: "
        f"{row['count']}개"
    )

# ============================================================
# 13. 연결 종료
# ============================================================

driver.close()

Neo4j 연결 성공
URI      : bolt://localhost:7687
Database : neo4j

데이터 적재 완료

[생성된 노드]


,label,name,crno,address
0,Industry,IT서비스,NaN,NaN
1,Industry,데이터서비스,NaN,NaN
2,Industry,에너지,NaN,NaN
3,Industry,자동차부품,NaN,NaN
4,ParentCompany,가온전자,1101110001002,경기도 성남시 분당구 판교로 210
5,ParentCompany,가온홀딩스,1101110001001,서울특별시 강남구 테헤란로 120
6,ParentCompany,새빛테크,1601110001003,대전광역시 유성구 테크노로 80
7,Region,경기도,NaN,NaN
8,Region,대전광역시,NaN,NaN
9,Region,서울특별시,NaN,NaN



[레이블별 노드 수]


,label,count
0,Industry,4
1,ParentCompany,3
2,Region,4
3,SubsidiaryCompany,4



[생성된 관계]


...


[관계 타입별 개수]


,relation,count
0,AFFILIATED_WITH,1
1,HAS_SUBSIDIARY,4
2,LOCATED_IN,7
3,RELATED_TO_INDUSTRY,4



[온톨로지 관계 검증]


,source,source_label,relation,target,target_label,valid
0,가온디지털,SubsidiaryCompany,LOCATED_IN,서울특별시,Region,True
1,가온디지털,SubsidiaryCompany,RELATED_TO_INDUSTRY,IT서비스,Industry,True
2,가온모빌리티,SubsidiaryCompany,LOCATED_IN,경기도,Region,True
3,가온모빌리티,SubsidiaryCompany,RELATED_TO_INDUSTRY,자동차부품,Industry,True
4,가온전자,ParentCompany,HAS_SUBSIDIARY,가온모빌리티,SubsidiaryCompany,True
5,가온전자,ParentCompany,LOCATED_IN,경기도,Region,True
6,가온홀딩스,ParentCompany,AFFILIATED_WITH,가온전자,ParentCompany,True
7,가온홀딩스,ParentCompany,HAS_SUBSIDIARY,가온디지털,SubsidiaryCompany,True
8,가온홀딩스,ParentCompany,LOCATED_IN,서울특별시,Region,True
9,새빛데이터,SubsidiaryCompany,LOCATED_IN,대전광역시,Region,True



적재 결과
확인된 노드 수 : 15
확인된 관계 수 : 16

모든 관계가 현재 온톨로지 규칙을 만족합니다.

[사용된 노드 레이블]
- Industry: 4개
- ParentCompany: 3개
- Region: 4개
- SubsidiaryCompany: 4개

[사용된 관계]
- AFFILIATED_WITH: 1개
- HAS_SUBSIDIARY: 4개
- LOCATED_IN: 7개
- RELATED_TO_INDUSTRY: 4개


적재 전, 초기화
기존 노드·관계·제약조건을 모두 지우고 처음부터 다시 적재하기 위한 초기화

In [ ]:
from neo4j import GraphDatabase

run_cypher("MATCH (n) DETACH DELETE n")

for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")

실제 데이터 기반 Neo4j 적재

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase


# 프로젝트 루트 찾기
# 두 단계 위를 프로젝트 루트로 사용
PROJECT_ROOT = Path.cwd().parents[1]

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# .env 읽기
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)


NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")


if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD가 설정되어 있지 않습니다.")


# Neo4j 연결
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)

driver.verify_connectivity()


def run_cypher(query, **params):
    """Cypher 실행 후 결과를 dict 리스트로 반환."""
    with driver.session() as session:
        return [
            record.data()
            for record in session.run(query, **params)
        ]

print("Neo4j 연결 성공:", NEO4J_URI)


# =========================================================


import json

from collections import defaultdict
from pathlib import Path


# =========================
# 파일 위치
# =========================

CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

NODE_JSONL_PATH = CLEAN_DIR / "기업관계_노드.jsonl"
TRIPLE_JSONL_PATH = CLEAN_DIR / "기업관계_트리플.jsonl"

BATCH_SIZE = 1000


# =========================
# 허용 스키마
# =========================

NODE_TYPES = {
    "ParentCompany",
    "SubsidiaryCompany",
    "Region",
    "Industry",
}


ALLOWED_SIGNATURES = {
    ("AFFILIATED_WITH", "ParentCompany", "ParentCompany"),
    ("HAS_SUBSIDIARY", "ParentCompany", "SubsidiaryCompany"),
    ("LOCATED_IN", "ParentCompany", "Region"),
    ("LOCATED_IN", "SubsidiaryCompany", "Region"),
    ("IN_INDUSTRY", "ParentCompany", "Industry"),
    ("IN_INDUSTRY", "SubsidiaryCompany", "Industry"),
}


# =========================
# JSONL 읽기
# =========================

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def run_in_batches(query, rows, batch_size=BATCH_SIZE):
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        run_cypher(query, rows=batch)


# 파일 존재 확인
if not NODE_JSONL_PATH.exists():
    raise FileNotFoundError(
        f"노드 파일을 찾을 수 없습니다: {NODE_JSONL_PATH}"
    )

if not TRIPLE_JSONL_PATH.exists():
    raise FileNotFoundError(
        f"트리플 파일을 찾을 수 없습니다: {TRIPLE_JSONL_PATH}"
    )


print("노드 파일:", NODE_JSONL_PATH)
print("트리플 파일:", TRIPLE_JSONL_PATH)


# =========================
# 1. 노드 읽기
# =========================

node_rows = read_jsonl(NODE_JSONL_PATH)

nodes_by_type = defaultdict(list)

for row in node_rows:

    node_type = row["type"]

    if node_type not in NODE_TYPES:
        raise ValueError(
            f"허용되지 않은 노드 타입: {node_type}"
        )

    nodes_by_type[node_type].append(row)


# =========================
# 2. ID 유일성 제약조건
# =========================

for node_type in NODE_TYPES:

    constraint_name = f"{node_type.lower()}_id_unique"

    query = f"""
    CREATE CONSTRAINT {constraint_name} IF NOT EXISTS
    FOR (n:{node_type})
    REQUIRE n.id IS UNIQUE
    """

    run_cypher(query)


# =========================
# 3. 노드 적재
# =========================

for node_type, rows in nodes_by_type.items():

    query = f"""
    UNWIND $rows AS row

    MERGE (n:{node_type} {{id: row.id}})

    SET n += row.properties
    SET n.id = row.id
    """

    run_in_batches(query, rows)


# =========================
# 4. 트리플 읽기
# =========================

triple_rows = read_jsonl(TRIPLE_JSONL_PATH)

triples_by_signature = defaultdict(list)

for row in triple_rows:

    signature = (
        row["relation"],
        row["subject_type"],
        row["object_type"],
    )

    if signature not in ALLOWED_SIGNATURES:
        raise ValueError(
            f"허용되지 않은 관계 시그니처: {signature}"
        )

    triples_by_signature[signature].append(row)


# =========================
# 5. 관계 적재
# =========================

for (
    relation,
    subject_type,
    object_type,
), rows in triples_by_signature.items():

    query = f"""
    UNWIND $rows AS row

    MATCH (subject:{subject_type} {{id: row.subject}})
    MATCH (object:{object_type} {{id: row.object}})

    MERGE (subject)-[r:{relation}]->(object)

    SET r.source_case = row.source_case,
        r.source_row = row.source_row,
        r.evidence = row.evidence
    """

    run_in_batches(query, rows)


# =========================
# 결과 확인
# =========================

print(f"노드 적재 대상: {len(node_rows):,}개")
print(f"트리플 적재 대상: {len(triple_rows):,}개")

print(
    "현재 Neo4j 노드:",
    run_cypher(
        "MATCH (n) RETURN count(n) AS count"
    )[0]["count"],
)

print(
    "현재 Neo4j 관계:",
    run_cypher(
        "MATCH ()-[r]->() RETURN count(r) AS count"
    )[0]["count"],
)

Neo4j 연결 성공: bolt://localhost:7687
노드 파일: c:\Users\Playdata\Desktop\EDU2\mle-01-p2-team3\data\clean\기업관계_노드.jsonl
트리플 파일: c:\Users\Playdata\Desktop\EDU2\mle-01-p2-team3\data\clean\기업관계_트리플.jsonl
노드 적재 대상: 8,024개
트리플 적재 대상: 12,281개
현재 Neo4j 노드: 8024
현재 Neo4j 관계: 12281
